# Matrix Multiplication
**Topic:** Linear Algebra for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** matrix multiplication as a grid of dot products, one for each output cell
- **Determine** whether two matrices can be multiplied by checking their shapes, and predict the output shape
- **Interpret** the forward pass through a neural network layer as a single matrix multiplication

> **Tip:** Start with the **step-through animation** in the widget. Press Next one step at a time and read which row of A and which column of B are highlighted before looking at which cell of C lights up. After three steps, try to predict the next one.

---
## How we got here

In *03: The Dot Product* we learned to combine two vectors into a single number. Matrix multiplication extends this idea: each cell in the output matrix is the dot product of one row of the first matrix with one column of the second. If you understand the dot product, you understand matrix multiplication, it is just many dot products organized into a grid.

---
## Why this matters for data science

Every forward pass through a neural network layer is a matrix multiplication. When 64 data points pass through a layer with 128 neurons, Python computes one matrix multiplication: a (64, features) matrix times a (features, 128) weight matrix, producing a (64, 128) output. No explicit loop. No manual dot products. One operation handles all 64 examples simultaneously. This is why GPU-accelerated training with large batch sizes is so effective: modern hardware is optimized for exactly this operation.

---
## Try it yourself

In [2]:
out = Output()

# A is 3×2, B is 2×4 — small enough to read clearly
_A_mm = np.array([[1.0, 2.0],
                  [3.0, 4.0],
                  [5.0, 6.0]])
_B_mm = np.array([[7.0, 8.0, 9.0, 10.0],
                  [11.0, 12.0, 13.0, 14.0]])
_C_mm = _A_mm @ _B_mm  # (3, 4)

_m, _k = _A_mm.shape
_, _n  = _B_mm.shape
_total = _m * _n

step_sl = IntSlider(value=0, min=0, max=_total - 1, step=1,
    description="Step:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="500px"))

def render(change=None):
    s  = step_sl.value
    ci = s // _n   # output row
    cj = s % _n    # output col

    # Build per-cell highlight arrays (row-major: [row][col])
    def _highlight(nr, nc, row_hi=None, col_hi=None, cell_hi=None, hi_color=None):
        return [[hi_color if (
                    (row_hi is not None and i == row_hi) or
                    (col_hi is not None and j == col_hi) or
                    (cell_hi is not None and i == cell_hi[0] and j == cell_hi[1])
                ) else "#f0f4ff"
                for j in range(nc)] for i in range(nr)]

    a_fill  = _highlight(_m, _k, row_hi=ci,  hi_color=PALETTE["primary"])
    b_fill  = _highlight(_k, _n, col_hi=cj,  hi_color=PALETTE["secondary"])
    c_fill  = _highlight(_m, _n, cell_hi=(ci, cj), hi_color=PALETTE["accent"])

    def _font(fill_grid):
        return [["white" if c != "#f0f4ff" else "#333"
                 for c in row] for row in fill_grid]

    # go.Table: values are column-major lists
    def _table(M, fill, header_vals, hi_col=None):
        nr2, nc2 = M.shape
        col_fill  = [[fill[r][c] for r in range(nr2)] for c in range(nc2)]
        col_font  = [[_font(fill)[r][c] for r in range(nr2)] for c in range(nc2)]
        col_vals  = [[M[r, c] for r in range(nr2)] for c in range(nc2)]
        return go.Table(
            columnwidth=[60]*nc2,
            header=dict(values=header_vals,
                fill_color=PALETTE["muted"],
                font=dict(color="white", size=11, family=FONT["family"]),
                align="center", height=28),
            cells=dict(values=col_vals,
                fill_color=col_fill,
                font=dict(color=col_font, size=12, family=FONT["family"]),
                align="center", height=34))

    fig = make_subplots(rows=1, cols=3,
        specs=[[{"type": "table"}, {"type": "table"}, {"type": "table"}]],
        subplot_titles=[f"A ({_m}×{_k})", f"B ({_k}×{_n})", f"C = A @ B ({_m}×{_n})"],
        column_widths=[0.20, 0.40, 0.40])

    fig.add_trace(_table(_A_mm, a_fill,
        [f"col {j}" for j in range(_k)]), row=1, col=1)
    fig.add_trace(_table(_B_mm, b_fill,
        [f"col {j}" for j in range(_n)]), row=1, col=2)
    fig.add_trace(_table(_C_mm, c_fill,
        [f"col {j}" for j in range(_n)]), row=1, col=3)

    terms   = " + ".join(f"{_A_mm[ci,l]:.0f}×{_B_mm[l,cj]:.0f}" for l in range(_k))
    formula = f"C[{ci},{cj}] = {terms} = {_C_mm[ci,cj]:.0f}"
    fig.update_layout(
        title=dict(
            text=f"Step {s+1}/{_total}: row {ci} of A · col {cj} of B  |  {formula}",
            font=dict(size=13, family=FONT["family"])),
        height=200 + _m * 36,
        paper_bgcolor=PALETTE["background"],
        margin=dict(t=55, b=10, l=10, r=10))

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

step_sl.observe(render, names="value")
display(VBox([step_sl, out]))
render()

---
## What's happening?

To multiply matrix **A** (shape m × k) by matrix **B** (shape k × n), the inner dimensions must match: **A** has k columns and **B** has k rows. The result **C** has shape m × n.

Each element C[i, j] is the dot product of row i of **A** with column j of **B**:

$$C_{ij} = \sum_{l=1}^{k} A_{il} \cdot B_{lj}$$

The total result matrix **C** is computed by doing this for every (i, j) pair.

$$\mathbf{C} = \mathbf{A} \mathbf{B} \quad \text{where } \mathbf{A}: (m \times k),\; \mathbf{B}: (k \times n) \Rightarrow \mathbf{C}: (m \times n)$$

| Input shapes | Condition | Output shape |
|-------------|-----------|-------------|
| (3, 4) @ (4, 2) | Inner dimensions match: 4 = 4 | (3, 2) |
| (100, 20) @ (20, 10) | Inner dimensions match: 20 = 20 | (100, 10) |
| (3, 4) @ (3, 2) | Inner mismatch: 4 ≠ 3 | Error |
| (n, p) @ (p, k) | Inner match: p = p | (n, k) |

### The neural network forward pass

A fully connected layer computes: **Z = X W + b**

- **X** is the input batch, shape (n, p), n examples, p features
- **W** is the weight matrix, shape (p, k), p input features, k neurons
- **b** is the bias vector, shape (k,), one bias per neuron
- **Z** is the output, shape (n, k), n examples, k activations

One matrix multiplication produces activations for every example in the batch simultaneously. This is the forward pass.

Step through the widget and compute a complete (3 × 2) @ (2 × 4) multiplication by hand, then verify your answer matches the widget's output.

### Matrix multiplication shape rules

| Input shapes | Valid? | Output shape | ML operation |
|-------------|--------|-------------|-------------|
| (n, p) @ (p, k) | Yes | (n, k) | Forward pass through one dense layer |
| (p, k) @ (k, n) | Yes | (p, n) | Weight transpose used in normal equations |
| (n, p) @ (n, k) | No: inner mismatch | Error | Common beginner mistake |
| (k,) @ (k, n) | Yes (broadcast) | (n,) | Single example through a layer |
| (n, p) @ (p,) | Yes (broadcast) | (n,) | Batch dot product with a weight vector |

---
## Key takeaway

> **Matrix multiplication produces a grid of dot products — each output cell is the inner product of one row of the first matrix with one column of the second — and this single operation computes a full neural network layer for every example in the batch at once.**

---
*Next up: Transpose & Inverse — two matrix properties that unlock the elegant closed-form solution to linear regression*